# 📈 07. Validation Evaluation & Error Analysis
### Amazon ML Challenge 2026: Multi-Source Business Entity Resolution

This notebook evaluates the end-to-end model on the held-out validation set:
1. **Confusion Matrix Analysis** (True Positives, False Positives, False Negatives, True Negatives).
2. **Singleton Accuracy & Failure Analysis**.
3. **Error Breakdown:** False merges (high-cost) vs Missed matches.
4. **Test Set Prediction Sample Inspection**.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("..").resolve() if Path("..").resolve().name == "amazon" else Path(".").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import json

import pandas as pd

from src.utils.io import load_config

cfg = load_config(PROJECT_ROOT / "config.yaml")
val_rep_path = PROJECT_ROOT / "artifacts" / "reports" / "validation_metrics.json"

if val_rep_path.exists():
    with open(val_rep_path, encoding="utf-8") as f:
        metrics = json.load(f)
    print("Validation Evaluation Metrics:")
    for k, v in metrics.items():
        if isinstance(v, float):
            print(f"  {k:<26}: {v:.4f}")
        else:
            print(f"  {k:<26}: {v}")

## 1. Validation Confusion Matrix Table
Display the pairwise confusion matrix.

In [ ]:
if val_rep_path.exists():
    tp = metrics.get("pair_tp", 0)
    fp = metrics.get("pair_fp", 0)
    fn = metrics.get("pair_fn", 0)

    cm_df = pd.DataFrame(
        [
            {"Prediction": "Match (Positive)", "Actual True Match": f"{tp:,} (TP)", "Actual Non-Match": f"{fp:,} (FP)"},
            {
                "Prediction": "Non-Match (Negative)",
                "Actual True Match": f"{fn:,} (FN)",
                "Actual Non-Match": "TN (Filtered by Blocker)",
            },
        ]
    )
    display(cm_df)

## 2. Test Predictions Inspection
Inspect sample predictions generated for the test set.

In [ ]:
matching_tsv = PROJECT_ROOT / cfg["paths"]["final_matching_file"]
if matching_tsv.exists():
    df_sub = pd.read_csv(matching_tsv, sep="\t", nrows=20, keep_default_na=False)
    print("Sample Submission Rows (First 20):")
    display(df_sub)